# Predicción de enlaces
## SP500

Sea $G=(V,E)$ una red de empresas, donde cada vértice representa una compañía del S&P 500 y cada arista representa una relación de dependencia, proveeduría o complementariedad comercial. La red original se construye como dirigida, con interpretación $u	o v$ cuando $u$ provee, vende o habilita algo para $v$.

Para aplicar PAC, Adamic-Adar y Jaccard se usa la proyección no dirigida del grafo. Esta transformación reemplaza la pregunta direccional por una pregunta estructural: dos empresas son cercanas si comparten vecinos, intermediarios o posiciones similares dentro de la minired. Por tanto, los resultados no deben leerse como predicciones financieras directas, sino como predicciones topológicas de enlaces plausibles.


In [1]:
import html
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import networkx as nx
from IPython.display import HTML, display


In [2]:
from typing import List, Tuple

V: List[str] = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "GOOG", "AVGO", "META", "TSLA", "BRK.B",
    "JPM", "V", "MA", "WMT", "COST", "ORCL", "LLY", "XOM", "JNJ", "PLTR",
    "BAC", "ABBV", "MU", "HD", "NFLX", "AMD", "PG", "GE", "CVX", "UNH"
]

E: List[Tuple[str, str]] = [
    ("MU", "NVDA"), ("AVGO", "AAPL"),
    ("NVDA", "MSFT"), ("NVDA", "AMZN"), ("NVDA", "GOOG"), ("NVDA", "GOOGL"),
    ("NVDA", "ORCL"), ("NVDA", "META"), ("NVDA", "TSLA"),
    ("AMD", "AMZN"), ("AMD", "MSFT"), ("AMD", "ORCL"), ("AMD", "TSLA"),
    ("AMZN", "NFLX"), ("AMZN", "XOM"), ("AMZN", "JNJ"), ("AMZN", "ABBV"), ("AMZN", "PLTR"),
    ("MSFT", "CVX"), ("MSFT", "GE"), ("MSFT", "UNH"), ("MSFT", "PLTR"),
    ("GOOGL", "PLTR"), ("ORCL", "PLTR"),
    ("AAPL", "WMT"), ("AAPL", "COST"), ("PG", "WMT"), ("BRK.B", "WMT"),
    ("V", "COST"), ("V", "HD"), ("V", "BAC"),
    ("MA", "HD"), ("MA", "WMT"), ("MA", "AAPL"), ("MA", "BAC"), ("JPM", "AAPL"),
    ("LLY", "AMZN"),
]

G_dir = nx.DiGraph()
G_dir.add_nodes_from(V)
G_dir.add_edges_from(E)

# Las tres tecnicas usadas aqui estan definidas para grafos no dirigidos en NetworkX.
# Por eso se proyecta la red proveedor -> comprador a una red simple: hay arista si existe relacion en cualquier direccion.
G1 = nx.Graph(G_dir)
print("Dirigida | Nodos:", G_dir.number_of_nodes(), "| Aristas:", G_dir.number_of_edges())
print("No dirigida | Nodos:", G1.number_of_nodes(), "| Aristas:", G1.number_of_edges())


Dirigida | Nodos: 30 | Aristas: 37
No dirigida | Nodos: 30 | Aristas: 37


In [3]:
def show_table(rows, columns=None, float_digits=4):
    if not rows:
        display(HTML("<em>Sin datos.</em>"))
        return

    if columns is None:
        columns = list(rows[0].keys())

    def fmt(value):
        if isinstance(value, float):
            return f"{value:.{float_digits}f}"
        return html.escape(str(value))

    header = "".join(f"<th>{html.escape(str(col))}</th>" for col in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{fmt(row.get(col, ''))}</td>" for col in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"""
    <table style="border-collapse:collapse; font-size:14px; max-width:100%;">
      <thead><tr style="background:#f2f2f2;">{header}</tr></thead>
      <tbody>{body}</tbody>
    </table>
    <style>
      table td, table th {{ border:1px solid #ddd; padding:5px 8px; vertical-align:top; }}
      table th {{ text-align:left; }}
    </style>
    """))


def graph_summary(G):
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    communities, _ = modularity_partition(G)
    return [{
        "nodos": G.number_of_nodes(),
        "aristas": G.number_of_edges(),
        "densidad": nx.density(G),
        "componentes": len(components),
        "tamano_componente_mayor": len(components[0]) if components else 0,
        "comunidades_modularidad": len(communities),
        "pares_sin_arista": len(list(nx.non_edges(G)))
    }]


def top_degrees(G, k=10):
    return [
        {"vertice": node, "grado": degree}
        for node, degree in sorted(G.degree, key=lambda item: item[1], reverse=True)[:k]
    ]


def modularity_partition(G):
    communities = list(nx.algorithms.community.greedy_modularity_communities(G))
    communities = [sorted(list(c), key=str) for c in communities]
    communities.sort(key=lambda c: (-len(c), str(c[0]) if c else ""))
    community_of = {}
    for idx, community in enumerate(communities):
        for node in community:
            community_of[node] = idx
    return communities, community_of


def prediction_table(G, method, k=10):
    """Devuelve las k aristas no observadas con mayor puntaje."""
    ebunch = list(nx.non_edges(G))
    communities, community_of = modularity_partition(G)
    if method == "PAC":
        rows = nx.preferential_attachment(G, ebunch)
    elif method == "AAC":
        rows = nx.adamic_adar_index(G, ebunch)
    elif method == "JAC":
        rows = nx.jaccard_coefficient(G, ebunch)
    else:
        raise ValueError("method debe ser PAC, AAC o JAC")

    rows = sorted(rows, key=lambda row: (row[2], str(row[0]), str(row[1])), reverse=True)[:k]
    out = []
    for u, v, score in rows:
        cu = community_of.get(u, -1)
        cv = community_of.get(v, -1)
        out.append({
            "u": u,
            "v": v,
            "puntaje": score,
            "comunidad_u": cu,
            "comunidad_v": cv,
            "tipo_modularidad": "intra" if cu == cv else "inter",
            "vecinos_comunes": ", ".join(sorted(nx.common_neighbors(G, u, v)))
        })
    return out


def _scale_positions(pos, width, height, margin):
    return _scale_positions_box(pos, width, height, margin, margin, margin, margin)


def _scale_positions_box(pos, width, height, left, right, top, bottom):
    xs = [xy[0] for xy in pos.values()]
    ys = [xy[1] for xy in pos.values()]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    span_x = max(max_x - min_x, 1e-9)
    span_y = max(max_y - min_y, 1e-9)
    return {
        node: (
            left + (xy[0] - min_x) / span_x * (width - left - right),
            top + (xy[1] - min_y) / span_y * (height - top - bottom),
        )
        for node, xy in pos.items()
    }


def modularity_seed_layout(G):
    communities, community_of = modularity_partition(G)
    pos = {}
    n_com = max(len(communities), 1)
    for ci, community in enumerate(communities):
        center_angle = 2 * math.pi * ci / n_com
        cx = 2.8 * math.cos(center_angle)
        cy = 2.8 * math.sin(center_angle)
        inner_n = max(len(community), 1)
        inner_radius = 0.42 + 0.055 * inner_n
        for j, node in enumerate(community):
            angle = 2 * math.pi * j / inner_n + 0.37 * ci
            pos[node] = (cx + inner_radius * math.cos(angle), cy + inner_radius * math.sin(angle))
    return pos, communities, community_of


def force_layout(G, iterations=180):
    pos, communities, community_of = modularity_seed_layout(G)
    nodes = list(G.nodes())
    n = max(len(nodes), 1)
    area = 32.0
    k = math.sqrt(area / n)
    edges = list(G.edges())

    for step in range(iterations):
        disp = {node: [0.0, 0.0] for node in nodes}
        temp = 0.22 * (1 - step / iterations) + 0.018

        for i, u in enumerate(nodes):
            ux, uy = pos[u]
            for v in nodes[i + 1:]:
                vx, vy = pos[v]
                dx = ux - vx
                dy = uy - vy
                dist = math.sqrt(dx * dx + dy * dy) + 1e-6
                force = (k * k) / dist
                fx = dx / dist * force
                fy = dy / dist * force
                disp[u][0] += fx
                disp[u][1] += fy
                disp[v][0] -= fx
                disp[v][1] -= fy

        for u, v in edges:
            ux, uy = pos[u]
            vx, vy = pos[v]
            dx = ux - vx
            dy = uy - vy
            dist = math.sqrt(dx * dx + dy * dy) + 1e-6
            same = community_of.get(u) == community_of.get(v)
            strength = 1.55 if same else 0.75
            force = strength * (dist * dist) / k
            fx = dx / dist * force
            fy = dy / dist * force
            disp[u][0] -= fx
            disp[u][1] -= fy
            disp[v][0] += fx
            disp[v][1] += fy

        for node in nodes:
            dx, dy = disp[node]
            length = math.sqrt(dx * dx + dy * dy) + 1e-6
            x, y = pos[node]
            pos[node] = (x + dx / length * min(length, temp), y + dy / length * min(length, temp))
    return pos, communities, community_of


def show_network_svg(G, title="Red", predicted=None, width=980, height=780, label_mode="auto"):
    predicted = predicted or []
    raw_pos, communities, community_of = force_layout(G)
    pos = _scale_positions_box(raw_pos, width, height, left=78, right=54, top=170, bottom=52)
    degrees = dict(G.degree())
    max_degree = max(degrees.values()) if degrees else 1
    palette = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759", "#76b7b2", "#edc948", "#b07aa1", "#ff9da7", "#9c755f", "#bab0ab"]

    predicted_nodes = {node for row in predicted for node in (row["u"], row["v"])}
    top_label_nodes = {node for node, _ in sorted(G.degree, key=lambda item: item[1], reverse=True)[:10]}
    subtitle = f"{G.number_of_nodes()} nodos | {G.number_of_edges()} aristas observadas | {len(communities)} comunidades por modularidad"

    community_parts = []
    for ci, community in enumerate(communities):
        xs = [pos[node][0] for node in community]
        ys = [pos[node][1] for node in community]
        if not xs:
            continue
        cx = sum(xs) / len(xs)
        cy = sum(ys) / len(ys)
        radius = max([math.sqrt((x - cx) ** 2 + (y - cy) ** 2) for x, y in zip(xs, ys)] + [22]) + 40
        color = palette[ci % len(palette)]
        community_parts.append(
            f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{radius:.1f}" fill="{color}" opacity="0.075" stroke="{color}" stroke-width="1.3" stroke-dasharray="5 5" />'
        )
        community_parts.append(
            f'<text x="{cx:.1f}" y="{max(cy - radius + 18, 158):.1f}" font-size="12" text-anchor="middle" font-family="Arial" fill="{color}">C{ci} ({len(community)} nodos)</text>'
        )

    edge_parts = []
    for u, v in G.edges():
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        same = community_of.get(u) == community_of.get(v)
        stroke = "#a9b1bb" if same else "#7d8793"
        opacity = "0.42" if same else "0.68"
        width_line = "1.0" if same else "1.6"
        edge_parts.append(
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{stroke}" stroke-width="{width_line}" opacity="{opacity}" />'
        )

    pred_parts = []
    if predicted:
        scores = [row["puntaje"] for row in predicted]
        min_s, max_s = min(scores), max(scores)
    else:
        min_s, max_s = 0, 1
    for row in predicted:
        u, v = row["u"], row["v"]
        if u not in pos or v not in pos:
            continue
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        score = row["puntaje"]
        same = community_of.get(u) == community_of.get(v)
        color = palette[community_of.get(u, 0) % len(palette)] if same else "#d62728"
        width_line = 2.4 + 3.2 * ((score - min_s) / (max_s - min_s + 1e-9))
        pred_parts.append(
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{color}" stroke-width="{width_line:.1f}" opacity="0.97">'
            f'<title>{html.escape(str(u))} - {html.escape(str(v))}: {score:.4f} | {"intra-comunidad" if same else "inter-comunidad"}</title></line>'
        )

    node_parts = []
    label_parts = []
    for node in G.nodes():
        x, y = pos[node]
        r = 7 + 14 * degrees[node] / max_degree
        color = palette[community_of.get(node, 0) % len(palette)]
        node_parts.append(
            f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" fill="{color}" stroke="#222" stroke-width="1.1">'
            f'<title>{html.escape(str(node))} | grado {degrees[node]} | comunidad C{community_of.get(node, -1)}</title></circle>'
        )
        should_label = label_mode == "all" or node in predicted_nodes or node in top_label_nodes
        if should_label:
            label = html.escape(str(node))
            label_parts.append(
                f'<text x="{x:.1f}" y="{y-r-5:.1f}" font-size="10" text-anchor="middle" font-family="Arial, sans-serif" fill="#1b1b1b">{label}</text>'
            )

    community_legend = []
    for ci, community in enumerate(communities[:6]):
        color = palette[ci % len(palette)]
        x = 565 + (ci % 3) * 122
        y = 88 + (ci // 3) * 26
        community_legend.append(
            f'<circle cx="{x:.1f}" cy="{y:.1f}" r="6" fill="{color}" stroke="#222" stroke-width="0.6" />'
            f'<text x="{x+12:.1f}" y="{y+4:.1f}" font-size="12" font-family="Arial" fill="#333">C{ci}: {len(community)}</text>'
        )

    svg = f"""
    <div style="max-width:{width}px; overflow-x:auto; font-family:Arial, sans-serif;">
      <svg viewBox="0 0 {width} {height}" width="100%" height="auto" role="img" aria-label="{html.escape(title)}">
        <title>{html.escape(title)}</title>
        <desc>{html.escape(subtitle)}. Los colores indican comunidades de modularidad; las lineas gruesas son enlaces predichos.</desc>
        <rect width="100%" height="100%" fill="#ffffff" />

        <text x="24" y="32" font-size="22" font-family="Arial, sans-serif" font-weight="700" fill="#111">{html.escape(title)}</text>
        <text x="24" y="56" font-size="13" font-family="Arial, sans-serif" fill="#555">{html.escape(subtitle)}</text>

        <rect x="20" y="70" width="930" height="54" rx="8" fill="#f8f9fb" stroke="#d6dbe1" stroke-width="1" />
        <text x="34" y="92" font-size="13" font-family="Arial" font-weight="700" fill="#222">Leyenda</text>
        <line x1="34" y1="110" x2="86" y2="110" stroke="#a9b1bb" stroke-width="1.6" opacity="0.8" />
        <text x="96" y="114" font-size="12" font-family="Arial" fill="#333">Coaparición</text>
        <line x1="220" y1="110" x2="272" y2="110" stroke="#4e79a7" stroke-width="4" />
        <text x="282" y="114" font-size="12" font-family="Arial" fill="#333">Predicción dentro de su comunidad</text>
        <line x1="545" y1="110" x2="597" y2="110" stroke="#d62728" stroke-width="4" />
        <text x="607" y="114" font-size="12" font-family="Arial" fill="#333">Predicción fuera de su comunidad</text>
        <circle cx="780" cy="110" r="5" fill="#4e79a7" stroke="#222" stroke-width="0.8" />
        <circle cx="798" cy="110" r="10" fill="#4e79a7" stroke="#222" stroke-width="0.8" />
        <text x="814" y="114" font-size="12" font-family="Arial" fill="#333">Tamaño del nodo = grado</text>

        <g>{''.join(community_parts)}</g>
        <g>{''.join(edge_parts)}</g>
        <g>{''.join(pred_parts)}</g>
        <g>{''.join(node_parts)}</g>
        <g>{''.join(label_parts)}</g>
      </svg>
    </div>
    """
    display(HTML(svg))


## Análisis de la red

La red contiene $|V|=30$ empresas y $|E|=37$ aristas en la proyección no dirigida. La densidad es

$$
\rho(G)=\frac{2|E|}{|V|(|V|-1)}\approx 0.085,
$$

lo cual indica una red dispersa. Solo una fracción pequeña de las relaciones posibles está presente.

La red tiene dos componentes conectadas. La componente principal agrupa semiconductores, nube, inteligencia artificial y compradores tecnológicos: NVDA, AMZN, MSFT, AMD, ORCL, GOOGL, PLTR, entre otros. La segunda componente agrupa consumo, pagos y retail alrededor de AAPL, MA, V, WMT, COST, HD y BAC.

Los grados más altos corresponden a NVDA y AMZN con 8 conexiones, MSFT con 6, AAPL con 5, y luego WMT, PLTR, MA y AMD con 4. La predicción se calcula sobre 398 pares sin arista. Como la red está separada en componentes, los métodos basados en vecinos comunes tienden a proponer enlaces dentro de componentes; PAC puede generar sugerencias por grado alto aun sin vecinos comunes.

Para interpretar comunidades se usa modularidad:

$$
Q=\frac{1}{2m}\sum_{i,j}\left(A_{ij}-\frac{k_i k_j}{2m}\right)\delta(c_i,c_j).
$$

La partición se obtiene con `greedy_modularity_communities`, que busca comunidades con más aristas internas de las esperadas bajo un modelo nulo con la misma secuencia de grados. Esta partición permite distinguir predicciones dentro de bloques económicos y predicciones que cruzan bloques.


In [4]:
show_table(graph_summary(G1))
show_table(top_degrees(G1, k=10))


nodos,aristas,densidad,componentes,tamano_componente_mayor,comunidades_modularidad,pares_sin_arista
30,37,0.0851,2,19,4,398


vertice,grado
NVDA,8
AMZN,8
MSFT,6
AAPL,5
MA,4
WMT,4
PLTR,4
AMD,4
V,3
ORCL,3


## Figura de la red base

La figura usa un layout tipo fuerza inicializado con comunidades de modularidad. Los colores representan comunidades; los círculos de fondo muestran la región aproximada de cada comunidad. El tamaño del nodo representa el grado $k_v$, es decir, cuántas conexiones tiene la empresa dentro de la minired.

En las figuras de PAC, AAC y JAC, una predicción dentro de comunidad toma el color de esa comunidad. Una predicción fuera de comunidad aparece en rojo. Esta convención visual permite distinguir si el método está cerrando relaciones dentro de un bloque económico o proponiendo enlaces entre bloques.


In [5]:
show_network_svg(G1, title="Red base con comunidades de modularidad")


## Resumen de las tres técnicas

Sea $\Gamma(u)$ el conjunto de vecinos de $u$ y sea $k_u=|\Gamma(u)|$ su grado. Cada técnica define un puntaje $s(u,v)$ para pares no observados $(u,v)\notin E$.

Emparejamiento Preferencial (PAC):

$$
s_{PAC}(u,v)=k_u k_v.
$$

Este índice favorece empresas con muchas relaciones observadas. En esta red, por construcción, tenderá a favorecer hubs como NVDA, AMZN, MSFT y AAPL.

Adamic-Adar (AAC):

$$
s_{AAC}(u,v)=\sum_{w\in\Gamma(u)\cap\Gamma(v)}\frac{1}{\log k_w}.
$$

Este índice mide vecinos comunes, pero da más peso a intermediarios poco genéricos. Es útil para detectar cierres de triángulos dentro de bloques de negocio.

Jaccard (JAC):

$$
s_{JAC}(u,v)=\frac{|\Gamma(u)\cap\Gamma(v)|}{|\Gamma(u)\cup\Gamma(v)|}.
$$

Este índice mide similitud proporcional. Puede ser alto para empresas periféricas que comparten el mismo único vecino, aunque la evidencia absoluta sea pequeña.

Así, PAC modela crecimiento por popularidad, AAC modela cierre por intermediarios informativos y JAC modela equivalencia de vecindario.


## Usando Emparejamiento Preferencial (PAC)

PAC favorece empresas con grado alto. El primer enlace sugerido es MSFT--AMZN con puntaje 48, porque MSFT tiene grado 6 y AMZN grado 8. En este caso el puntaje alto también tiene soporte estructural, ya que comparten AMD, NVDA y PLTR.

Sin embargo, PAC también sugiere enlaces como NVDA--AAPL, AAPL--AMZN, WMT--AMZN, NVDA--WMT, NVDA--PLTR, NVDA--MA y NVDA--AMD. Algunos de estos pares no comparten vecinos en la proyección no dirigida; aparecen porque sus extremos tienen alto grado. Esto ilustra la principal limitación del índice: puede confundir importancia marginal con cercanía estructural.

Desde la modularidad, PAC puede producir predicciones dentro y fuera de comunidades. Las intra-comunidad se interpretan como refuerzo de bloques económicos existentes. Las inter-comunidad, marcadas en rojo, deben leerse con cautela: pueden indicar puentes potenciales, pero también pueden ser artefactos del producto de grados.


In [6]:
pac = prediction_table(G1, "PAC", k=10)
show_table(pac)
show_network_svg(G1, title="Enlaces predichos con PAC", predicted=pac[:8])


u,v,puntaje,comunidad_u,comunidad_v,tipo_modularidad,vecinos_comunes
AMZN,MSFT,48,2,3,inter,"AMD, NVDA, PLTR"
AMZN,AAPL,40,2,0,inter,
AAPL,NVDA,40,0,1,inter,
WMT,NVDA,32,0,1,inter,
PLTR,NVDA,32,1,1,intra,"AMZN, GOOGL, MSFT, ORCL"
NVDA,MA,32,1,0,inter,
AMZN,WMT,32,2,0,inter,
AMZN,MA,32,2,0,inter,
AMD,NVDA,32,1,1,intra,"AMZN, MSFT, ORCL, TSLA"
AAPL,MSFT,30,0,3,inter,


## Usando el Coeficiente Adamic-Adar (AAC)

AAC da un ranking más estructural. Los primeros enlaces son NVDA--AMD y NVDA--PLTR, ambos con puntaje aproximado 3.392. En NVDA--AMD los vecinos comunes son AMZN, MSFT, ORCL y TSLA; en NVDA--PLTR son AMZN, GOOGL, MSFT y ORCL. Estos pares comparten intermediarios dentro del bloque de inteligencia artificial, nube y cómputo.

También aparece MA--V con puntaje 2.885, compartiendo BAC y HD. Este caso es relevante porque MA y V no necesitan ser los nodos de mayor grado para aparecer arriba; basta con que compartan intermediarios relativamente informativos dentro del bloque financiero/retail.

En términos de modularidad, AAC tiende a generar predicciones intra-comunidad. Cuando genera una predicción inter-comunidad, la evidencia es más robusta que en PAC porque existe un conjunto explícito de vecinos comunes. Por eso AAC es especialmente adecuado para detectar relaciones latentes dentro de bloques funcionales de la red.


In [7]:
aac = prediction_table(G1, "AAC", k=10)
show_table(aac)
show_network_svg(G1, title="Enlaces predichos con AAC", predicted=aac[:8])


u,v,puntaje,comunidad_u,comunidad_v,tipo_modularidad,vecinos_comunes
PLTR,NVDA,3.3919,1,1,intra,"AMZN, GOOGL, MSFT, ORCL"
AMD,NVDA,3.3919,1,1,intra,"AMZN, MSFT, ORCL, TSLA"
V,MA,2.8854,0,0,intra,"BAC, HD"
PLTR,AMD,1.9492,1,1,intra,"AMZN, MSFT, ORCL"
ORCL,MSFT,1.9236,1,3,inter,"AMD, NVDA, PLTR"
AMZN,ORCL,1.9236,2,1,inter,"AMD, NVDA, PLTR"
AMZN,MSFT,1.9236,2,3,inter,"AMD, NVDA, PLTR"
BAC,HD,1.6316,0,0,intra,"MA, V"
AAPL,V,1.4427,0,0,intra,COST
TSLA,ORCL,1.2022,1,1,intra,"AMD, NVDA"


## Usando el Coeficiente Jaccard (JAC)

Jaccard resalta pares con vecindarios proporcionalmente idénticos. Por eso varios enlaces tienen puntaje 1.0: XOM--NFLX, XOM--LLY, XOM--ABBV, NFLX--LLY, JNJ--XOM y JNJ--NFLX comparten exclusivamente a AMZN. También UNH--CVX comparten exclusivamente a MSFT, y MU--META comparten exclusivamente a NVDA.

La interpretación debe ser cuidadosa. Un puntaje $s_{JAC}=1$ no implica que las empresas sean similares en sentido económico amplio; implica que, dentro de esta minired, sus vecindarios observados coinciden proporcionalmente. Cuando los grados son bajos, esta coincidencia puede producir puntajes perfectos con poca evidencia absoluta.

Por modularidad, JAC tiende a cerrar hojas dentro de una misma comunidad o estrella local. Esto revela equivalencia estructural periférica: empresas que ocupan posiciones parecidas alrededor de un mismo intermediario. Es útil para detectar roles locales, pero menos adecuado para priorizar enlaces estratégicos entre hubs.


In [8]:
jac = prediction_table(G1, "JAC", k=10)
show_table(jac)
show_network_svg(G1, title="Enlaces predichos con JAC", predicted=jac[:8])


u,v,puntaje,comunidad_u,comunidad_v,tipo_modularidad,vecinos_comunes
XOM,NFLX,1.0000,2,2,intra,AMZN
XOM,LLY,1.0000,2,2,intra,AMZN
XOM,JNJ,1.0000,2,2,intra,AMZN
XOM,ABBV,1.0000,2,2,intra,AMZN
UNH,CVX,1.0000,3,3,intra,MSFT
PG,BRK.B,1.0000,0,0,intra,WMT
NFLX,LLY,1.0000,2,2,intra,AMZN
MU,GOOG,1.0000,1,1,intra,NVDA
META,MU,1.0000,1,1,intra,NVDA
META,GOOG,1.0000,1,1,intra,NVDA


## Análisis específico de resultados

En SP500 las diferencias entre técnicas son más marcadas que en Star Wars porque la red es más dispersa y está separada en bloques económicos. PAC queda dominado por el grado y recomienda relaciones alrededor de NVDA, AMZN, MSFT y AAPL. Algunas, como MSFT--AMZN, tienen soporte por vecinos comunes; otras son más débiles porque aparecen solo por popularidad.

AAC produce el ranking más convincente para esta red. Enlaces como NVDA--AMD, NVDA--PLTR, MA--V, AMD--PLTR, ORCL--MSFT, ORCL--AMZN y MSFT--AMZN cierran triángulos dentro de bloques económicos claros. Su ventaja es que combina proximidad local con una penalización a intermediarios demasiado generales.

JAC funciona como detector de equivalencia local. Muchos nodos periféricos solo tienen un vecino; si dos comparten ese único vecino, Jaccard vale 1. Esto explica enlaces como XOM--NFLX o UNH--CVX. El resultado es matemáticamente correcto, pero debe interpretarse como similitud de posición en la minired, no como evidencia fuerte de una relación económica directa.

La modularidad permite evaluar el sentido de las predicciones. Las líneas del color de la comunidad indican cierre interno del bloque; las líneas rojas indican conexiones entre bloques. En conjunto, AAC es la técnica más informativa para inferir enlaces dentro de comunidades de negocio, PAC describe expansión de hubs y JAC revela equivalencias locales en la periferia.
